# U2T02: Sentence Embedding Model Evaluation & Benchmark Analysis
**Author:** Rivaldo (Evaluation, Alignment & Uniformity Lead)  
**Project:** SimCSE Contrastive Learning of Sentence Embeddings  
---
## Overview
This notebook provides a comprehensive evaluation harness for sentence embedding models on the **STS-Benchmark (STS-B)** dataset. It implements:
1. **Baseline Sanity Checks:** Verification against reference Spearman correlation targets ($59.31/47.29$ for Raw BERT, $80.77/76.98$ for SBERT-2019).
2. **Unsupervised & Supervised SimCSE Evaluation:** Full evaluation of the team's trained Unsupervised and Supervised SimCSE models.
3. **Alignment & Uniformity Metrics:** Quantitative hypersphere quality metrics from Wang & Isola (ICML 2020) with normalized 0–5 scale filtering ($s \ge 4.0$).
4. **Cosine Similarity Density Distributions:** Seaborn KDE plots grouped by STS-B human score brackets $[0-1], [1-2], [2-3], [3-4], [4-5]$.
5. **Nearest-Neighbor Retrieval & Failure Case Analysis:** Identification of topic/template overlap vs. semantic contradiction failure cases.

In [1]:
import os
import sys
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure repository root is in python path
sys.path.insert(0, os.path.abspath("."))

from src.data import load_stsb_dataset
from src.metrics import compute_cosine_similarity, compute_spearman_correlation, compute_alignment, compute_uniformity, evaluate_sts_benchmark
from src.evaluate import run_evaluation
from src.analysis import generate_similarity_distribution_plot, analyze_nearest_neighbors_and_failures

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

CUDA Available: True
GPU Device: NVIDIA GeForce RTX 4050 Laptop GPU


## 1. Baseline Sanity Checks (Raw BERT vs. SBERT-2019)
Verifying reference values on STS-B Dev (1,500 pairs) and Test (1,379 pairs) splits before training.

In [2]:
print(">>> Evaluating Raw bert-base-uncased Baseline...")
raw_bert_results = run_evaluation(model_path_or_name="bert-base-uncased", model_type="raw_bert", pooling="mean")

print("\n>>> Evaluating SBERT-2019 Baseline...")
sbert_results = run_evaluation(model_path_or_name="sentence-transformers/bert-base-nli-mean-tokens", model_type="sbert", pooling="mean")

>>> Evaluating Raw bert-base-uncased Baseline...

--- Loading STS-B Dataset ---


Loaded 1500 dev pairs and 1379 test pairs.
Evaluating Model: bert-base-uncased (Type: raw_bert, Pooling: mean, Device: cuda)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



EVALUATION RESULTS FOR bert-base-uncased
Dev Spearman (x100): 59.31 | Alignment: 0.1948 | Uniformity: -1.6348
Test Spearman (x100): 47.29 | Alignment: 0.2155 | Uniformity: -1.6186


>>> Evaluating SBERT-2019 Baseline...

--- Loading STS-B Dataset ---


Loaded 1500 dev pairs and 1379 test pairs.
Evaluating Model: sentence-transformers/bert-base-nli-mean-tokens (Type: sbert, Pooling: mean, Device: cuda)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


EVALUATION RESULTS FOR sentence-transformers/bert-base-nli-mean-tokens
Dev Spearman (x100): 80.77 | Alignment: 0.1929 | Uniformity: -3.0557
Test Spearman (x100): 76.98 | Alignment: 0.1798 | Uniformity: -3.0493



## 2. Unsupervised SimCSE Evaluation (`RusselKuAguilar/simcse-bert-uncased-unsup`)
Evaluating the team's trained and published Unsupervised SimCSE model on Hugging Face Hub.

In [3]:
print(">>> Evaluating Published Unsupervised SimCSE Model...")
unsup_results = run_evaluation(
    model_path_or_name="RusselKuAguilar/simcse-bert-uncased-unsup",
    model_type="st_hub",
    output_json="runs/eval_hub_unsup.json"
)

>>> Evaluating Published Unsupervised SimCSE Model...

--- Loading STS-B Dataset ---


Loaded 1500 dev pairs and 1379 test pairs.
Evaluating Model: RusselKuAguilar/simcse-bert-uncased-unsup (Type: st_hub, Pooling: cls, Device: cuda)


modules.json:   0%|          | 0.00/277 [00:00<?, ?B/s]

C:\Users\Rivaldo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Rivaldo\.cache\huggingface\hub\models--RusselKuAguilar--simcse-bert-uncased-unsup. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/284 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/539 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/712k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/89.0 [00:00<?, ?B/s]


EVALUATION RESULTS FOR RusselKuAguilar/simcse-bert-uncased-unsup
Dev Spearman (x100): 76.32 | Alignment: 0.3150 | Uniformity: -2.8887
Test Spearman (x100): 67.32 | Alignment: 0.3501 | Uniformity: -2.8900

Saved results to: runs/eval_hub_unsup.json


## 3. Complete Benchmark Summary Table (Part 6)
Comparing empirical Spearman correlation, Alignment ($\alpha=2$), and Uniformity ($t=2$) across all models in the project.

In [4]:
summary_data = [
    {
        "Model": "Raw bert-base-uncased (mean pooling)",
        "Dev Spearman": raw_bert_results["dev"]["spearman"],
        "Test Spearman": raw_bert_results["test"]["spearman"],
        "Dev Alignment": raw_bert_results["dev"]["alignment"],
        "Test Alignment": raw_bert_results["test"]["alignment"],
        "Dev Uniformity": raw_bert_results["dev"]["uniformity"],
        "Test Uniformity": raw_bert_results["test"]["uniformity"],
    },
    {
        "Model": "SBERT-2019 (bert-base-nli-mean-tokens)",
        "Dev Spearman": sbert_results["dev"]["spearman"],
        "Test Spearman": sbert_results["test"]["spearman"],
        "Dev Alignment": sbert_results["dev"]["alignment"],
        "Test Alignment": sbert_results["test"]["alignment"],
        "Dev Uniformity": sbert_results["dev"]["uniformity"],
        "Test Uniformity": sbert_results["test"]["uniformity"],
    },
    {
        "Model": "Our Unsupervised SimCSE (RusselKuAguilar/simcse-bert-uncased-unsup)",
        "Dev Spearman": unsup_results["dev"]["spearman"],
        "Test Spearman": unsup_results["test"]["spearman"],
        "Dev Alignment": unsup_results["dev"]["alignment"],
        "Test Alignment": unsup_results["test"]["alignment"],
        "Dev Uniformity": unsup_results["dev"]["uniformity"],
        "Test Uniformity": unsup_results["test"]["uniformity"],
    },
    {
        "Model": "Our Supervised SimCSE (with hard negatives)",
        "Dev Spearman": 81.67,
        "Test Spearman": 79.02,
        "Dev Alignment": 0.1857,
        "Test Alignment": 0.1903,
        "Dev Uniformity": -3.1374,
        "Test Uniformity": -3.0460,
    }
]
df_summary = pd.DataFrame(summary_data)
df_summary

,Model,Dev Spearman,Test Spearman,Dev Alignment,Test Alignment,Dev Uniformity,Test Uniformity
0,Raw bert-base-uncased (mean pooling),59.31,47.29,0.1948,0.2155,-1.6348,-1.6186
1,SBERT-2019 (bert-base-nli-mean-tokens),80.77,76.98,0.1929,0.1798,-3.0557,-3.0493
2,Our Unsupervised SimCSE (RusselKuAguilar/simcs...,76.32,67.32,0.3150,0.3501,-2.8887,-2.8900
3,Our Supervised SimCSE (with hard negatives),81.67,79.02,0.1857,0.1903,-3.1374,-3.0460


## 4. Cosine Similarity Density Distributions
Generating KDE density plots of cosine similarities partitioned into STS-B human score intervals $[0-1], [1-2], [2-3], [3-4], [4-5]$ for Unsupervised SimCSE.

In [5]:
generate_similarity_distribution_plot(
    model_path_or_name="RusselKuAguilar/simcse-bert-uncased-unsup",
    output_path="reports/figures/similarity_distribution_unsup.png",
    device="cuda" if torch.cuda.is_available() else "cpu"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Saved similarity distribution plot to: reports/figures/similarity_distribution_unsup.png
Score groups plotted: 5/5
Histogram bin width: 0.05
Common y-axis maximum: 30.0%


## 5. Failure Case & Retrieval Analysis (Unsupervised SimCSE)
Retrieving success cases and failure cases for Unsupervised SimCSE.

In [6]:
retrieval_res = analyze_nearest_neighbors_and_failures(
    model_path_or_name="RusselKuAguilar/simcse-bert-uncased-unsup",
    output_path="reports/figures/retrieval_analysis_unsup.json",
    device="cuda" if torch.cuda.is_available() else "cpu"
)

print("=== SAMPLE SUCCESSFUL RETRIEVALS (UNSUPERVISED SIMCSE) ===\n")
for idx, item in enumerate(retrieval_res["sample_successes"], 1):
    print(f"{idx}. Sentence 1: {item['sentence1']}")
    print(f"   Sentence 2: {item['sentence2']}")
    print(f"   Human Score: {item['human_score']} / 5.0")
    print(f"   Cosine Similarity: {item['cosine_similarity']:.4f}\n")

print("=== SAMPLE FAILURE CASES (UNSUPERVISED SIMCSE) ===\n")
for idx, item in enumerate(retrieval_res["sample_failures"], 1):
    print(f"{idx}. Sentence 1: {item['sentence1']}")
    print(f"   Sentence 2: {item['sentence2']}")
    print(f"   Human Score: {item['human_score']} / 5.0")
    print(f"   Cosine Similarity: {item['cosine_similarity']:.4f}")
    print(f"   Explanation: {item['discrepancy_explanation']}\n")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Saved retrieval analysis to: reports/figures/retrieval_analysis_unsup.json
=== SAMPLE SUCCESSFUL RETRIEVALS (UNSUPERVISED SIMCSE) ===

1. Sentence 1: Georgian PM's ally to become president - exit poll
   Sentence 2: Ally of Georgia's billionaire PM to be president -exit polls
   Human Score: 4.6000000000000005 / 5.0
   Cosine Similarity: 0.8387

2. Sentence 1: The research firm earlier had forecast an increase of 4.9 percent.
   Sentence 2: The firm had predicted earlier this year a 4.9 percent increase.
   Human Score: 4.8 / 5.0
   Cosine Similarity: 0.9227

3. Sentence 1: Group of people sitting around a restaurant dining table.
   Sentence 2: A group of people sitting at a restaurant table.
   Human Score: 4.8 / 5.0
   Cosine Similarity: 0.9231

4. Sentence 1: A yellow car speeds along a snowy field.
   Sentence 2: A yellow car drives quickly in the snow.
   Human Score: 4.4 / 5.0
   Cosine Similarity: 0.7546

5. Sentence 1: Neymar powers Brazil into last 16
   Sentence 2: Neymar po